# CEG-WM：相位敏感性与 VAE 重编码影响的最小诊断

**复用保存的 Oracle RGB，不重新生成图像。只加载原版 VAE，不加载去噪器、文本编码器或几何预测器。**

沿用成功的 **Python 3.12 GPU 环境**和 Secrets：`HF_TOKEN`、`CEG_WM_ROOT_KEY`，运行全部单元。
只读输入：`MyDrive/CEG-WM/crop-bottleneck-v1/attempt01/`；新输出：`MyDrive/CEG-WM/crop-mechanism-v1/attempt01/`。

固定 8 对图像 × 7 个整数平移条件 = **112 条记录**。先重放 16 个原 Oracle 分数，任何一张不匹配即停止。
本地只验证过工程与数学对照，尚未运行真实 GPU 诊断；本 notebook 不预填结论。

## 对照设计

RGB 平移：零位移、x/y 各 ±2 像素，以及 x/y 各 +8 像素。数组直接复制，不做 RGB 插值。
8 RGB 像素对应一个潜变量格点；2 RGB 像素对应四分之一格点。

设 `z=E(x)`、`b=E(全黑画布)`、`r=Q(z-b)`。Q 去除 64×64 偶数网格的 Nyquist 行/列，
避免分数格点 Fourier 平移后丢弃虚部而误称“纯相位”。所有反事实 HF 比较使用同一去 Nyquist 掩膜。

1. `b + FourierShift(r)`：周期边界、保范数且可逆的纯相位对照。
2. 零扩展 32/64 格后的 Fourier 平移：检查边界模型和填充收敛。
3. 周期／零扩展双线性潜变量平移：检查插值模型差异。
4. `E(整数平移 RGB)`：实际 VAE 重编码路径；黑填充与循环移位分别检查，像素相同则复用编码。
5. 对实际重编码潜变量逆移已知位移：检查正确坐标补偿后是否仍有额外偏差。
6. 不减黑画布基线的周期相位对照：检查基线建模是否左右结论。

原始 LF/HF/联合分数和历史阈值只用于实际 RGB 读出。反事实 HF 只报告连续分数和正确钥减最大错钥的间隔，
**不沿用原联合阈值、不选最优位移、不宣称盲检测性能**。


In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import os, pathlib, subprocess, sys, json, re, time
SOURCE = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-bottleneck-v1/attempt01')
RUN_ID = 'attempt01'
assert re.fullmatch(r'[A-Za-z0-9_-]+', RUN_ID)
output = pathlib.Path('/content/drive/MyDrive/CEG-WM/crop-mechanism-v1') / RUN_ID
checkout = pathlib.Path('/content/cegwm-crop-mechanism-source')
script_dir = pathlib.Path('/content/cegwm-crop-mechanism')
script_dir.mkdir(exist_ok=True)
script = script_dir / 'run_mechanism.py'
expected = 'd9ef4adb4f8424f1102f577bb76347fbba112bd9'
saved_env = json.loads((SOURCE / 'environment.json').read_text())
saved_protocol = json.loads((SOURCE / 'protocol.json').read_text())
saved_summary = json.loads((SOURCE / 'summary.json').read_text())
if saved_summary['status'] != 'COMPLETE_DIAGNOSTIC':
    raise RuntimeError('Original crop run is incomplete. No generation will be started.')
source_python = tuple(map(int, saved_env['python'].split('.')[:2]))
if sys.version_info[:2] != source_python:
    raise RuntimeError(f'Use the successful historical Colab Python {source_python[0]}.{source_python[1]} runtime; current Python is {sys.version.split()[0]}. No dependencies or source results were changed.')
child_env = dict(os.environ)
for name in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
    value = userdata.get(name)
    if not value:
        raise RuntimeError(f'Missing secret: {name}')
    child_env[name] = value
child_env['HF_HOME'] = '/content/cegwm-crop-v1-hf'
child_env['HF_HUB_CACHE'] = child_env['HF_HOME'] + '/hub'
child_env['PYTHONUNBUFFERED'] = '1'
child_env.pop('HF_HUB_OFFLINE', None)
child_env.pop('TRANSFORMERS_OFFLINE', None)

def run_logged(command, label, **kwargs):
    # Explicit PIPE+print: Colab may hide output inherited by subprocess.run.
    log_dir = output / 'execution_logs'
    log_dir.mkdir(parents=True, exist_ok=True)
    log = log_dir / f'{label}-{time.time_ns()}.log'
    with log.open('x', encoding='utf-8') as sink:
        with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                              text=True, bufsize=1, **kwargs) as process:
            for line in process.stdout:
                for key in ('HF_TOKEN', 'CEG_WM_ROOT_KEY'):
                    line = line.replace(child_env[key], '[REDACTED]')
                print(line, end='', flush=True)
                sink.write(line)
                sink.flush()
            code = process.wait()
    if code:
        raise RuntimeError(f'{label} failed with exit code {code}. Full output: {log}')

run_logged([sys.executable, '-c',
    'import torch; assert torch.cuda.is_available(), "Select a GPU runtime"; '
    'print({"python":__import__("sys").version,"gpu":torch.cuda.get_device_name(0),"torch":torch.__version__})'], 'gpu-check')
print('Read-only source:', SOURCE)
print('New output:', output)


In [ ]:
# Match the successful source reader dependencies; preserve installed CUDA torch.
packages = saved_env['packages']
names = ('diffusers','transformers','accelerate','numpy','Pillow','huggingface_hub','safetensors','sentencepiece')
dependencies = [f'{name}=={packages[name]}' for name in names]
torch_versions = subprocess.run([sys.executable, '-c',
    'import importlib.metadata as m,json; print(json.dumps({p:m.version(p) for p in ("torch","torchvision")}))'],
    capture_output=True, text=True, check=True)
constraints = pathlib.Path('/content/cegwm-mechanism-cuda-constraints.txt')
constraints.write_text(''.join(f'{name}=={ver}\n' for name,ver in json.loads(torch_versions.stdout).items()))
run_logged([sys.executable, '-m', 'pip', 'install', '--only-binary=sentencepiece',
            '-c', str(constraints), *dependencies], 'pip-install')
if not checkout.exists():
    run_logged(['git', 'clone', 'https://github.com/RICHAAARC/CEG-WM.git', str(checkout)], 'clone')
run_logged(['git', '-C', str(checkout), 'checkout', '--detach', expected], 'checkout')
child_env['PYTHONPATH'] = str(checkout / 'src') + os.pathsep + str(checkout)
run_logged([sys.executable, '-c',
    'from diffusers import AutoencoderKL; from diffusers.image_processor import VaeImageProcessor; '
    'from cegwm.runtime.blind_scoring_v2 import _lf,_hf; '
    'print("VAE and frozen scoring imports OK; no models loaded")'], 'imports', env=child_env)


In [ ]:
# Embedded, version-bound reader and mechanism script.
(script_dir / 'run_residual.py').write_text('"""Read-only saved-image diagnostic. No generator, denoiser, or SyncSeal inference."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom dataclasses import asdict\nimport hashlib\nfrom importlib.metadata import version\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport uuid\n\nEXACT = "d9ef4adb4f8424f1102f577bb76347fbba112bd9"\nMODEL = "stabilityai/stable-diffusion-3.5-medium"\nTAU = 1.5545224858704942\nROLES = ("positive", "negative")\nIDS = tuple(f"v2-formal_evaluation_pairs-{i:04d}" for i in range(1, 9))\nATOL, RTOL = 1e-5, 1e-4\n\n\ndef read(path):\n    return json.loads(Path(path).read_text(encoding="utf-8"))\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as f:\n        for part in iter(lambda: f.read(1024 * 1024), b""):\n            h.update(part)\n    return h.hexdigest()\n\n\ndef write(path, data):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    tmp = path.with_name(path.name + "." + uuid.uuid4().hex + ".tmp")\n    with tmp.open("x", encoding="utf-8") as f:\n        json.dump(data, f, ensure_ascii=False, indent=2, allow_nan=False)\n        f.flush()\n        os.fsync(f.fileno())\n    os.replace(tmp, path)\n\n\ndef bind(path, data):\n    if Path(path).exists():\n        if read(path) != data:\n            raise RuntimeError(f"Resume identity mismatch: {Path(path).name}; do not mix runs")\n    else:\n        write(path, data)\n\n\ndef error_data(error):\n    trace = "".join(traceback.format_exception(error))\n    for name in ("HF_TOKEN", "CEG_WM_ROOT_KEY"):\n        if os.environ.get(name):\n            trace = trace.replace(os.environ[name], "[REDACTED]")\n    return {"error_type": type(error).__name__, "traceback_redacted": trace}\n\n\ndef cases():\n    result = [{"id": "oracle_replay", "kind": "oracle"},\n              {"id": "predicted_replay", "kind": "predicted"},\n              {"id": "oracle_uncorrected_pixel_convention", "kind": "convention"},\n              {"id": "half_prediction_residual", "kind": "half_residual"}]\n    for axis in ("x", "y"):\n        for value in (-4., -2., -.5, .5, 2., 4.):\n            result.append({"id": f"translation_{axis}_{value:+g}px", "kind": "translation",\n                           "axis": axis, "value": value})\n    for value in (-.04, -.02, -.005, .005, .02, .04):\n        result.append({"id": f"scale_{value * 100:+g}pct", "kind": "scale", "value": value})\n    return result\n\n\ndef normalization():\n    import numpy as np\n    return np.array([[2 / 511, 0., -1.], [0., 2 / 511, -1.], [0., 0., 1.]])\n\n\ndef matrix_for(case, crop, predicted):\n    """All perturbations compose BEFORE one sampling from saved attacked RGB."""\n    import numpy as np\n    base = np.asarray(crop["sampler_h_observed_to_canonical"], dtype=float)\n    kind = case["kind"]\n    if kind == "oracle":\n        return base\n    if kind == "predicted":\n        return np.asarray(predicted, dtype=float)\n    if kind == "convention":\n        return np.asarray(crop["physical_h_observed_to_canonical"], dtype=float)\n    if kind == "half_residual":\n        # A single predeclared diagnostic point along the actual predicted-H error.\n        # No interpolation weight is selected using detection scores.\n        p = np.asarray(predicted, dtype=float)\n        return .5 * base / base[2, 2] + .5 * p / p[2, 2]\n    residual = np.eye(3)\n    if kind == "translation":\n        residual[0 if case["axis"] == "x" else 1, 2] = case["value"]\n    elif kind == "scale":\n        scale = 1 + case["value"]\n        residual[0, 0] = residual[1, 1] = scale\n        residual[0, 2] = residual[1, 2] = 256 * (1 - scale)\n    else:\n        raise ValueError("Unknown fixed diagnostic case")\n    n = normalization()\n    return n @ residual @ np.linalg.inv(n) @ base\n\n\ndef geometry_metrics(matrix, crop):\n    import numpy as np\n    n = normalization()\n    edge = np.linalg.inv(n) @ matrix @ n\n    plus = np.array([[1., 0., .5], [0., 1., .5], [0., 0., 1.]])\n    center = np.linalg.inv(plus) @ edge @ plus\n    actual = n @ center @ np.linalg.inv(n)\n    truth = np.asarray(crop["physical_h_observed_to_canonical"])\n    points = np.array([[-1., -1., 1.], [1., -1., 1.], [1., 1., 1.], [-1., 1., 1.], [0., 0., 1.]]).T\n    a, b = actual @ points, truth @ points\n    delta = (a[:2] / a[2] - b[:2] / b[2]) * 511 / 2\n    return {"effective_corner_rmse_px": float(np.sqrt(np.mean(np.sum(delta[:, :4] ** 2, axis=0)))),\n            "effective_center_shift_xy_px": delta[:, 4].tolist()}\n\n\ndef load_saved_image(source, artifact):\n    from PIL import Image\n    path = Path(artifact["path"]).resolve()\n    if not path.is_relative_to(source.resolve()):\n        raise RuntimeError("Saved image points outside SOURCE directory")\n    if sha(path) != artifact["sha256"]:\n        raise RuntimeError(f"Saved image hash differs: {path.name}")\n    with Image.open(path) as image:\n        if image.mode != "RGB" or image.size != (512, 512):\n            raise ValueError("Saved input must be RGB 512x512")\n        return image.copy()\n\n\ndef inspect_source(source):\n    protocol, summary = read(source / "protocol.json"), read(source / "summary.json")\n    if summary["status"] != "COMPLETE_DIAGNOSTIC" or protocol["source_exact"] != EXACT:\n        raise ValueError("Completed original crop diagnostic required")\n    if tuple(u["unit_id"] for u in protocol["roster"]) != IDS or protocol["reference_tau"] != TAU:\n        raise ValueError("Source roster or threshold differs")\n    indexed = {(r["unit_id"], r["role"], r["route"]): r for r in summary["records"]}\n    if len(indexed) != 64 or len(summary["records"]) != 64:\n        raise ValueError("Expected 64 unique source route records")\n    for uid in IDS:\n        for role in ROLES:\n            for route in ("clean", "crop_direct", "oracle_h", "predicted_h"):\n                row = indexed[uid, role, route]\n                if row["status"] != "SCORED":\n                    raise ValueError("This fixed diagnostic requires all saved baseline routes")\n                load_saved_image(source, row["image"])\n    return protocol, indexed\n\n\nclass Reader:\n    """Only the VAE plus the unchanged LF/HF arithmetic from the pinned source."""\n    def __init__(self, repo, source):\n        import torch\n        from diffusers import AutoencoderKL\n        from diffusers.image_processor import VaeImageProcessor\n        from cegwm.method.lf import (FrozenLFPublicAssets, LF_BALANCED_BLOCKS_CARRIER_METHOD_ID,\n            LF_BLOCKNORM_DETECTOR_STATISTIC_ID, LF_BALANCED_BLOCKS_EVALUATED_CANDIDATE_ID)\n        from cegwm.method.hf import FrozenHFPublicAssets\n        from cegwm.method.content_whitening import FrozenContentWhiteningLFPublicAssets\n        from experiments.run_blind_detection_v1 import load_whitening_asset_semantic, load_weighted_asset_semantic\n        from cegwm.shared.keys import normalize_detection_key\n        from cegwm.runtime.content_weighted_joint_sd35 import derive_stability_wrong_keys\n        revision = read(source / "model_revisions.json")[MODEL]\n        self.vae = AutoencoderKL.from_pretrained(MODEL, subfolder="vae", revision=revision,\n            torch_dtype=torch.float16, token=os.environ["HF_TOKEN"]).to("cuda").eval()\n        # Exactly StableDiffusion3Pipeline v0.39.0\'s image_processor construction.\n        self.processor = VaeImageProcessor(vae_scale_factor=2 ** (len(self.vae.config.block_out_channels) - 1))\n        lf = FrozenLFPublicAssets(self.vae, self.processor, MODEL + ":image_processor",\n            LF_BALANCED_BLOCKS_CARRIER_METHOD_ID, LF_BLOCKNORM_DETECTOR_STATISTIC_ID,\n            LF_BALANCED_BLOCKS_EVALUATED_CANDIDATE_ID)\n        self.lf = FrozenContentWhiteningLFPublicAssets(lf, load_whitening_asset_semantic(repo))\n        self.hf = FrozenHFPublicAssets(self.vae, self.processor, MODEL + ":image_processor")\n        self.weighted = load_weighted_asset_semantic(repo)\n        self.key = normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])\n        self.keys = (self.key, *derive_stability_wrong_keys(self.key))\n\n    def observe(self, image):\n        from cegwm.runtime.observation import encode_final_rgb_image\n        return encode_final_rgb_image(image, self.processor, self.vae)\n\n    def score(self, observation):\n        from cegwm.runtime.blind_scoring_v2 import _lf, _hf\n        from cegwm.method.content_weighted_joint import weighted_joint_score\n        from cegwm.method.blind_detection import statistic_from_weighted_scores\n        labels = ("registered", *(f"wrong_{i:02d}" for i in range(16)))\n        pairs = [(_lf(observation, key, self.lf), _hf(observation, key, self.hf)) for key in self.keys]\n        branches = {"lf": {label: p[0] for label, p in zip(labels, pairs)},\n                    "hf": {label: p[1] for label, p in zip(labels, pairs)},\n                    "weighted_joint": {label: weighted_joint_score(*p, self.weighted) for label, p in zip(labels, pairs)}}\n        stat = statistic_from_weighted_scores(branches["weighted_joint"])\n        return {"branches": branches, "statistic": asdict(stat), "margin": stat.value - TAU,\n                "above_reference_tau": stat.value > TAU}\n\n\ndef replay_comparison(current, old):\n    import numpy as np\n    diffs, passed = {}, True\n    for branch in ("lf", "hf", "weighted_joint"):\n        a = list(current["branches"][branch].values())\n        b = list(old["branches"][branch].values())\n        diffs[branch] = float(np.max(np.abs(np.asarray(a) - b)))\n        passed &= bool(np.allclose(a, b, atol=ATOL, rtol=RTOL))\n    passed &= bool(np.isclose(current["statistic"]["value"], old["statistic"]["value"], atol=ATOL, rtol=RTOL))\n    return {"passed": passed, "max_abs_branch_differences": diffs,\n            "statistic_difference": current["statistic"]["value"] - old["statistic"]["value"],\n            "atol": ATOL, "rtol": RTOL}\n\n\ndef observation_metrics(image, oracle_image, observation, oracle_observation):\n    import numpy as np\n    a, b = np.asarray(image, dtype=float) / 255, np.asarray(oracle_image, dtype=float) / 255\n    z, z0 = observation.detach().float().cpu().numpy(), oracle_observation.detach().float().cpu().numpy()\n    def relative(x, y):\n        return float(np.linalg.norm(x - y) / max(float(np.linalg.norm(y)), 1e-12))\n    return {"rgb_rmse_full": float(np.sqrt(np.mean((a-b)**2))),\n            "rgb_rmse_fixed_inner": float(np.sqrt(np.mean((a[64:448,64:448]-b[64:448,64:448])**2))),\n            "vae_relative_l2_full": relative(z, z0),\n            "vae_relative_l2_fixed_inner": relative(z[...,8:56,8:56], z0[...,8:56,8:56])}\n\n\ndef summarize(output):\n    rows, counts = [], {}\n    for uid in IDS:\n        for role in ROLES:\n            for case in cases():\n                path = output / "records" / uid / role / (case["id"] + ".json")\n                row = read(path) if path.exists() else {"status": "PENDING"}\n                rows.append({**row, "unit_id": uid, "role": role, "case": case["id"]})\n    for role in ROLES:\n        for case in cases():\n            subset = [r for r in rows if r["role"] == role and r["case"] == case["id"]]\n            scored = [r for r in subset if r["status"] == "SCORED"]\n            counts[f"{role}/{case[\'id\']}"] = {"planned": 8, "scored": len(scored),\n                "above_reference_tau": sum(r.get("above_reference_tau", False) for r in scored),\n                "operational_error": sum(r["status"] == "OPERATIONAL_ERROR" for r in subset),\n                "pending": sum(r["status"] == "PENDING" for r in subset),\n                "mean_margin": sum(r["margin"] for r in scored) / len(scored) if scored else None}\n    result = {"status": "COMPLETE_DIAGNOSTIC" if all(r["status"] == "SCORED" for r in rows) else "INCOMPLETE",\n              "source_pairs": 8, "cases_per_image": len(cases()), "planned_scores": len(rows),\n              "claim_ceiling": "fixed_grid_sensitivity_diagnosis_not_blind_recovery_or_calibrated_FPR",\n              "counts": counts, "records": rows}\n    write(output / "summary.json", result)\n    cols = ["unit_id", "role", "case", "status", "margin", "delta_score_vs_oracle",\n            "delta_registered_vs_oracle", "delta_wrongmax_vs_oracle", "delta_hf_vs_oracle",\n            "delta_lf_vs_oracle", "oracle_above_reference_tau", "lost_vs_oracle", "effective_corner_rmse_px",\n            "rgb_rmse_full", "rgb_rmse_fixed_inner", "vae_relative_l2_full", "vae_relative_l2_fixed_inner"]\n    tmp = output / "readout.csv.tmp"\n    with tmp.open("w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=cols)\n        writer.writeheader()\n        writer.writerows({k: r.get(k) for k in cols} for r in rows)\n    os.replace(tmp, output / "readout.csv")\n    return result\n\n\ndef run(source, output, repo):\n    import numpy as np\n    import torch\n    from cegwm.geometry_v7.r1b import rectify_attacked_rgb\n    from cegwm.shared.keys import normalize_detection_key, public_key_digest\n    if output.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must be separate non-overlapping directories")\n    protocol, old = inspect_source(source)  # Read-only verification BEFORE any inference.\n    if subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip() != EXACT:\n        raise ValueError("Historical source revision differs")\n    if subprocess.check_output(["git", "-C", str(repo), "diff", "HEAD", "--"], text=True):\n        raise ValueError("Tracked historical source has modifications")\n    if public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"])) != protocol["key_digest"]:\n        raise ValueError("Detection key differs from saved run")\n    identity = {"source": str(source.resolve()), "source_protocol_sha": sha(source / "protocol.json"),\n                "source_summary_sha": sha(source / "summary.json"), "source_model_revisions_sha": sha(source / "model_revisions.json"),\n                "source_environment_sha": sha(source / "environment.json"), "script_sha": sha(__file__),\n                "code_exact": EXACT, "cases": cases(), "tau_reference_only": TAU, "atol": ATOL, "rtol": RTOL}\n    bind(output / "protocol.json", identity)\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA VAE reader required; no image generation fallback")\n    env = {"python": platform.python_version(), "gpu": torch.cuda.get_device_name(0), "cuda": torch.version.cuda,\n           "packages": {p: version(p) for p in ("torch", "torchvision", "diffusers", "transformers", "numpy", "Pillow", "huggingface_hub", "safetensors")}}\n    bind(output / "environment.json", env)\n    if summarize(output)["status"] == "COMPLETE_DIAGNOSTIC":\n        print("Already complete; no scoring repeated.", flush=True)\n        return\n    reader = Reader(repo, source)\n    crop = protocol["crop"]\n    references = {}\n\n    def process(uid, role, case, attacked, oracle_image, oracle_z, oracle_score):\n        path = output / "records" / uid / role / (case["id"] + ".json")\n        if path.exists() and read(path)["status"] == "SCORED":\n            return read(path)\n        attempt = output / "attempts" / uid / role / case["id"] / (uuid.uuid4().hex + ".json")\n        write(attempt, {"status": "STARTED", "time": time.time()})\n        try:\n            predicted = old[uid, role, "predicted_h"]["h_passed_to_frozen_renderer"]\n            matrix = matrix_for(case, crop, predicted)\n            image = rectify_attacked_rgb(attacked, matrix.tolist())\n            if case["kind"] in ("oracle", "predicted"):\n                previous = old[uid, role, "oracle_h" if case["kind"] == "oracle" else "predicted_h"]\n                if not np.array_equal(np.asarray(image), np.asarray(load_saved_image(source, previous["image"]))):\n                    raise RuntimeError("Replay RGB differs from saved route; stop before residual tests")\n            z = oracle_z if case["kind"] == "oracle" else reader.observe(image)\n            scored = reader.score(z)\n            replay = replay_comparison(scored, previous) if case["kind"] in ("oracle", "predicted") else None\n            if replay is not None and not replay["passed"]:\n                write(attempt, {"status": "REPLAY_MISMATCH", "replay": replay, **scored})\n                raise RuntimeError("VAE-only reader replay mismatch; residual diagnosis halted")\n            a, b = scored["statistic"], oracle_score["statistic"]\n            result = {"status": "SCORED", "h_passed_to_renderer": matrix.tolist(), **scored,\n                      **geometry_metrics(matrix, crop), **observation_metrics(image, oracle_image, z, oracle_z),\n                      "replay": replay, "delta_score_vs_oracle": a["value"] - b["value"],\n                      "delta_registered_vs_oracle": a["registered_weighted_joint"] - b["registered_weighted_joint"],\n                      "delta_wrongmax_vs_oracle": a["wrong_key_max"] - b["wrong_key_max"],\n                      "delta_hf_vs_oracle": scored["branches"]["hf"]["registered"] - oracle_score["branches"]["hf"]["registered"],\n                      "delta_lf_vs_oracle": scored["branches"]["lf"]["registered"] - oracle_score["branches"]["lf"]["registered"],\n                      "oracle_above_reference_tau": oracle_score["above_reference_tau"],\n                      "lost_vs_oracle": oracle_score["above_reference_tau"] and not scored["above_reference_tau"]}\n            write(attempt, result)\n            write(path, result)\n            print(f"{uid[-4:]} {role} {case[\'id\']}: margin={scored[\'margin\']:.4f}", flush=True)\n            return result\n        except Exception as error:\n            result = {"status": "OPERATIONAL_ERROR", **error_data(error)}\n            # Keep detailed replay values if available, alongside the final error.\n            if attempt.exists() and read(attempt).get("status") == "REPLAY_MISMATCH":\n                result["replay_failure"] = read(attempt)\n            write(attempt, result)\n            write(path, result)\n            summarize(output)\n            raise\n\n    # Complete ALL 32 replay controls before starting any perturbation inference.\n    for uid in IDS:\n        for role in ROLES:\n            attacked = load_saved_image(source, old[uid, role, "crop_direct"]["image"])\n            oracle_image = load_saved_image(source, old[uid, role, "oracle_h"]["image"])\n            oracle_z = reader.observe(oracle_image)\n            oracle_score = reader.score(oracle_z)\n            check = replay_comparison(oracle_score, old[uid, role, "oracle_h"])\n            if not check["passed"]:\n                write(output / "replay_gate.json", {"status": "FAILED", "unit_id": uid, "role": role, "check": check})\n                raise RuntimeError("Oracle reader replay mismatch; no residual tests started")\n            references[uid, role] = (attacked, oracle_image, oracle_z, oracle_score)\n            for case in cases()[:2]:\n                process(uid, role, case, *references[uid, role])\n            summarize(output)\n    write(output / "replay_gate.json", {"status": "PASSED", "planned": 32, "atol": ATOL, "rtol": RTOL})\n    for uid in IDS:\n        for role in ROLES:\n            for case in cases()[2:]:\n                process(uid, role, case, *references[uid, role])\n            summarize(output)\n    result = summarize(output)\n    print(json.dumps({"status": result["status"], "counts": result["counts"]}, indent=2), flush=True)\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--source", type=Path, required=True)\n    p.add_argument("--output", type=Path, required=True)\n    p.add_argument("--repo", type=Path, required=True)\n    args = p.parse_args()\n    # Guard this even for error reporting: never create a file inside SOURCE.\n    if args.output.resolve().is_relative_to(args.source.resolve()) or args.source.resolve().is_relative_to(args.output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    sys.path[:0] = [str(args.repo), str(args.repo / "src")]\n    try:\n        run(args.source, args.output, args.repo)\n    except Exception as error:\n        write(args.output / "last_failure.json", error_data(error))\n        raise\n\n\nif __name__ == "__main__":\n    main()\n', encoding='utf-8')
script.write_text('"""Saved-RGB mechanism probes. Counterfactual latent scores are NOT blind decisions."""\nfrom __future__ import annotations\nimport argparse\nimport csv\nfrom importlib.metadata import version\nimport math\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\nimport uuid\n\n# Same validated VAE-only reader and read-only source verifier as the previous run.\nimport run_residual as support\nfrom run_residual import (Reader, EXACT, MODEL, IDS, ROLES, ATOL, RTOL, read, write,\n    sha, bind, error_data, inspect_source, load_saved_image, replay_comparison)\n\nSUPPORT_SHA = "27945052b071b4a207d674c60cfd79f8d326448a09f5bdb3f54524cc31c1c701"\nSHIFTS = ((0,0),(-2,0),(2,0),(0,-2),(0,2),(8,0),(0,8))\n\n\ndef shift_id(dx,dy):\n    return f"dx{dx:+d}_dy{dy:+d}"\n\n\ndef rgb_shift(image, dx, dy, periodic=False):\n    """Exact uint8 permutation/copy. No RGB interpolation or PIL resampling."""\n    import numpy as np\n    from PIL import Image\n    arr=np.asarray(image)\n    shifted=np.roll(arr,(dy,dx),axis=(0,1)).copy()\n    if not periodic:\n        if dx>0: shifted[:,:dx]=0\n        if dx<0: shifted[:,dx:]=0\n        if dy>0: shifted[:dy]=0\n        if dy<0: shifted[dy:]=0\n    return Image.fromarray(shifted)\n\n\ndef project(z):\n    """Common real subspace without even-grid Nyquist row/column."""\n    import torch\n    f=torch.fft.fft2(z.float(),norm="ortho")\n    h,w=z.shape[-2:]\n    if h%2==0: f[...,h//2,:]=0\n    if w%2==0: f[...,:,w//2]=0\n    return torch.fft.ifft2(f,norm="ortho").real\n\n\ndef phase_shift(z, dx, dy):\n    """Unitary periodic Fourier translation on odd grids or the common subspace."""\n    import torch\n    h,w=z.shape[-2:]\n    f=torch.fft.fft2(z.float(),norm="ortho")\n    fy=torch.fft.fftfreq(h,device=z.device)[:,None]\n    fx=torch.fft.fftfreq(w,device=z.device)[None,:]\n    phase=torch.exp((-2j*math.pi)*(fx*dx+fy*dy))\n    shifted=torch.fft.ifft2(f*phase,norm="ortho")\n    # Do not silently throw away imaginary Nyquist energy on even grids.\n    if float(shifted.imag.norm()) > 2e-5*max(float(shifted.real.norm()),1e-12):\n        raise ValueError("Fourier shift left the real common subspace")\n    return shifted.real\n\n\ndef padded_phase(z, dx, dy, pad=32):\n    import torch.nn.functional as F\n    # 64+2*pad+1 is odd: no ambiguous Nyquist mode in the extension grid.\n    h,w=z.shape[-2:]\n    extended=F.pad(z,(pad,pad+1,pad,pad+1),mode="constant",value=0.)\n    moved=phase_shift(extended,dx,dy)\n    return moved[...,pad:pad+h,pad:pad+w]\n\n\ndef integer_roll(z, dx, dy, periodic):\n    import torch\n    result=torch.roll(z,(dy,dx),dims=(-2,-1)).clone()\n    if not periodic:\n        if dx>0: result[...,:,:dx]=0\n        if dx<0: result[...,:,dx:]=0\n        if dy>0: result[...,:dy,:]=0\n        if dy<0: result[...,dy:,:]=0\n    return result\n\n\ndef bilinear_shift(z, dx, dy, periodic):\n    ix,iy=math.floor(dx),math.floor(dy)\n    ax,ay=dx-ix,dy-iy\n    return sum(weight*integer_roll(z,ix+i,iy+j,periodic)\n               for i,j,weight in ((0,0,(1-ax)*(1-ay)),(1,0,ax*(1-ay)),\n                                  (0,1,(1-ax)*ay),(1,1,ax*ay)) if weight)\n\n\ndef relative(a,b):\n    return float((a-b).float().norm()/b.float().norm().clamp_min(1e-12))\n\n\ndef remove_fraction(z):\n    import torch\n    f=torch.fft.fft2(z.float(),norm="ortho")\n    removed=f.clone()\n    removed[...,z.shape[-2]//2,:]=0\n    removed[...,:,z.shape[-1]//2]=0\n    return float(((f-removed).abs()**2).sum()/(f.abs()**2).sum().clamp_min(1e-20))\n\n\nclass HFProbe:\n    """Original real-spectrum normalized cosine, full and fixed non-Nyquist masks."""\n    def __init__(self, reader, shape, device):\n        import torch\n        from cegwm.method.hf import _spec\n        from cegwm.method.frequency import radial_frequency_mask,reconstruct_frequency_carrier\n        spec=_spec(reader.hf)\n        h,w=shape[-2:]\n        full=torch.from_numpy(radial_frequency_mask(h,w,spec)).to(device)\n        common=full.clone()\n        common[h//2,:]=False\n        common[:,w//2]=False\n        self.masks={"full":full,"common":common}\n        self.templates={}\n        self.labels=("registered",*(f"wrong_{i:02d}" for i in range(16)))\n        spectra=[torch.fft.rfft2(reconstruct_frequency_carrier(key,shape,spec,dtype=torch.float32,\n                    device=device),norm="ortho").real for key in reader.keys]\n        for mode,mask in self.masks.items():\n            vectors=[]\n            for f in spectra:\n                v=f[...,mask].reshape(-1).double()\n                v=v-v.mean()\n                vectors.append(v/v.norm())\n            self.templates[mode]=torch.stack(vectors)\n\n    def score(self,z,mode="common"):\n        import torch\n        f=torch.fft.rfft2(z.float(),norm="ortho").real\n        v=f[...,self.masks[mode]].reshape(-1).double()\n        v=v-v.mean()\n        norm=v.norm()\n        if not torch.isfinite(norm) or float(norm)==0:\n            raise RuntimeError("Undefined HF diagnostic cosine")\n        values=(self.templates[mode]@(v/norm)).tolist()\n        if not all(math.isfinite(x) for x in values):\n            raise RuntimeError("Nonfinite diagnostic score")\n        return {"scores":dict(zip(self.labels,values)),"registered":values[0],\n                "wrong_max":max(values[1:]),"gap":values[0]-max(values[1:])}\n\n\ndef operator_fields(z0,black,z_actual,z_circular,dx,dy):\n    """Translate residual relative to encoded black; black baseline stays fixed."""\n    r=project(z0-black)\n    ax,ay=dx/8,dy/8\n    phase=phase_shift(r,ax,ay)\n    roundtrip=relative(phase_shift(phase,-ax,-ay),r)\n    if roundtrip>2e-5:\n        raise RuntimeError("Fourier phase round-trip failed")\n    actual=project(z_actual-black)\n    circular=project(z_circular-black)\n    fields={\n        "base":black+r,\n        "phase_periodic":black+phase,\n        "phase_pad32":black+padded_phase(r,ax,ay,32),\n        "phase_pad64":black+padded_phase(r,ax,ay,64),\n        "bilinear_periodic":black+bilinear_shift(r,ax,ay,True),\n        "bilinear_zero":black+bilinear_shift(r,ax,ay,False),\n        "actual_zero":black+actual,\n        "actual_circular":black+circular,\n        "actual_inverse_periodic":black+phase_shift(circular,-ax,-ay),\n        "actual_inverse_pad32":black+padded_phase(actual,-ax,-ay,32),\n        "phase_periodic_unsubtracted":phase_shift(project(z0),ax,ay),\n    }\n    expected=project(fields["phase_pad32"]-black)\n    metrics={"phase_roundtrip_relative_l2":roundtrip,\n             "phase_periodic_energy_ratio":float(phase.float().norm()/r.float().norm().clamp_min(1e-12)),\n             "nyquist_removed_energy_fraction_base_residual":remove_fraction(z0-black),\n             "nyquist_removed_energy_fraction_actual_residual":remove_fraction(z_actual-black),\n             "extra_relative_l2_full":relative(actual,expected),\n             "extra_relative_l2_inner":relative(actual[...,8:56,8:56],expected[...,8:56,8:56]),\n             "padding_convergence_relative_l2":relative(fields["phase_pad32"]-black,fields["phase_pad64"]-black)}\n    return fields,metrics\n\n\ndef decompose(scores):\n    """Signed telescoping score changes, NOT causal variance percentages."""\n    result={}\n    for metric in ("registered","gap"):\n        s={name:value[metric] for name,value in scores.items()}\n        result[metric]={\n            "total_loss":s["base"]-s["actual_zero"],\n            "phase_loss":s["base"]-s["phase_periodic"],\n            "boundary_model_delta":s["phase_periodic"]-s["phase_pad32"],\n            "extra_vs_padded_phase":s["phase_pad32"]-s["actual_zero"],\n            "bilinear_model_delta":s["phase_pad32"]-s["bilinear_zero"],\n            "extra_vs_bilinear":s["bilinear_zero"]-s["actual_zero"],\n            "periodic_interpolation_delta":s["phase_periodic"]-s["bilinear_periodic"],\n            "padding_32_vs_64":s["phase_pad32"]-s["phase_pad64"],\n            "rgb_boundary_delta":s["actual_circular"]-s["actual_zero"],\n            "black_baseline_model_delta":s["phase_periodic"]-s["phase_periodic_unsubtracted"],\n            "inverse_periodic_gain":s["actual_inverse_periodic"]-s["actual_circular"],\n            "inverse_pad32_gain":s["actual_inverse_pad32"]-s["actual_zero"],\n            "inverse_pad32_shortfall":s["base"]-s["actual_inverse_pad32"],\n        }\n    return result\n\n\ndef summarize(output):\n    rows=[]\n    for uid in IDS:\n        for role in ROLES:\n            for dx,dy in SHIFTS:\n                path=output/"records"/uid/role/(shift_id(dx,dy)+".json")\n                r=read(path) if path.exists() else {"status":"PENDING"}\n                rows.append({**r,"unit_id":uid,"role":role,"dx_rgb":dx,"dy_rgb":dy})\n    result={"status":"COMPLETE_DIAGNOSTIC" if all(r["status"]=="SCORED" for r in rows) else "INCOMPLETE",\n            "source_pairs":8,"planned_records":112,"records":rows,\n            "claim_ceiling":"oracle_translation_mechanism_probe_no_blind_detector_or_formal_FPR_claim"}\n    counts={}\n    for role in ROLES:\n        for dx,dy in SHIFTS:\n            rs=[r for r in rows if r["role"]==role and (r["dx_rgb"],r["dy_rgb"])==(dx,dy)]\n            ok=[r for r in rs if r["status"]=="SCORED"]\n            counts[role+"/"+shift_id(dx,dy)]={"planned":8,"scored":len(ok),\n                "pending":sum(r["status"]=="PENDING" for r in rs),\n                "operational_error":sum(r["status"]=="OPERATIONAL_ERROR" for r in rs),\n                "actual_production_above_reference_tau":sum(r["actual_production"]["above_reference_tau"] for r in ok)}\n    result["counts"]=counts\n    write(output/"summary.json",result)\n    flat=[]\n    for r in rows:\n        f={k:r[k] for k in ("unit_id","role","dx_rgb","dy_rgb","status")}\n        if r["status"]=="SCORED":\n            f.update(r["metrics"])\n            f["rgb_periodic_zero_identical"]=r["rgb_periodic_zero_identical"]\n            f["actual_production_margin"]=r["actual_production"]["margin"]\n            f["hf_full_actual"]=r["full_hf_actual"]["registered"]\n            f["hf_full_base"]=r["full_hf_base"]["registered"]\n            f["hf_common_actual"]=r["hf_diagnostic"]["actual_zero"]["registered"]\n            f["hf_common_base"]=r["hf_diagnostic"]["base"]["registered"]\n            f["hf_full_registered_loss"]=f["hf_full_base"]-f["hf_full_actual"]\n            f["hf_full_gap_loss"]=r["full_hf_base"]["gap"]-r["full_hf_actual"]["gap"]\n            for metric,changes in r["decomposition"].items():\n                f.update({metric+"_"+k:v for k,v in changes.items()})\n        flat.append(f)\n    columns=list(dict.fromkeys(k for r in flat for k in r))\n    tmp=output/"readout.csv.tmp"\n    with tmp.open("w",encoding="utf-8",newline="") as stream:\n        writer=csv.DictWriter(stream,fieldnames=columns)\n        writer.writeheader()\n        writer.writerows(flat)\n    os.replace(tmp,output/"readout.csv")\n    return result\n\n\ndef run(source,output,repo):\n    import numpy as np\n    import torch\n    from PIL import Image\n    from cegwm.shared.keys import normalize_detection_key,public_key_digest\n    if output.resolve().is_relative_to(source.resolve()) or source.resolve().is_relative_to(output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    if sha(support.__file__)!=SUPPORT_SHA:\n        raise ValueError("Validated VAE-only support module identity differs")\n    protocol,old=inspect_source(source)\n    if subprocess.check_output(["git","-C",str(repo),"rev-parse","HEAD"],text=True).strip()!=EXACT:\n        raise ValueError("Historical source revision differs")\n    if subprocess.check_output(["git","-C",str(repo),"diff","HEAD","--"],text=True):\n        raise ValueError("Historical tracked source modified")\n    if public_key_digest(normalize_detection_key(os.environ["CEG_WM_ROOT_KEY"]))!=protocol["key_digest"]:\n        raise ValueError("Key differs from source run")\n    identity={"schema":"crop_mechanism_v1","source":str(source.resolve()),"script_sha":sha(__file__),\n              "support_sha":SUPPORT_SHA,"source_exact":EXACT,"shifts_rgb":[list(s) for s in SHIFTS],\n              "source_hashes":{name:sha(source/name) for name in ("protocol.json","summary.json","environment.json","model_revisions.json")},\n              "boundary":"encoded_black_residual_circular_vs_zero_extension",\n              "phase":"Nyquist_row_column_removed_common_HF_subspace",\n              "padding_latent":[32,64],"inner_latent":[8,56],"replay_atol":ATOL,"replay_rtol":RTOL}\n    bind(output/"protocol.json",identity)\n    if not torch.cuda.is_available():\n        raise RuntimeError("CUDA VAE reader required; no generation fallback")\n    environment={"python":platform.python_version(),"gpu":torch.cuda.get_device_name(0),"cuda":torch.version.cuda,\n        "packages":{p:version(p) for p in ("torch","torchvision","diffusers","transformers","numpy","Pillow","huggingface_hub","safetensors")}}\n    bind(output/"environment.json",environment)\n    if summarize(output)["status"]=="COMPLETE_DIAGNOSTIC":\n        print("Already complete; no scoring repeated.",flush=True)\n        return\n    reader=Reader(repo,source)\n    black=reader.observe(Image.new("RGB",(512,512),(0,0,0))).float()\n    if tuple(black.shape)!=(1,16,64,64):\n        raise ValueError("Unexpected VAE observation shape")\n    probe=HFProbe(reader,tuple(black.shape),black.device)\n    refs={}\n    # All 16 saved Oracle baselines must replay before the first mechanism probe.\n    for uid in IDS:\n        for role in ROLES:\n            image=load_saved_image(source,old[uid,role,"oracle_h"]["image"])\n            z=reader.observe(image)\n            scored=reader.score(z)\n            check=replay_comparison(scored,old[uid,role,"oracle_h"])\n            full=probe.score(z,"full")\n            hf_ok=np.allclose(list(full["scores"].values()),list(scored["branches"]["hf"].values()),atol=ATOL,rtol=RTOL)\n            if not check["passed"] or not hf_ok:\n                write(output/"replay_gate.json",{"status":"FAILED","unit_id":uid,"role":role,"check":check,"cached_HF_matches":bool(hf_ok)})\n                raise RuntimeError("Baseline/HF scoring replay failed; no mechanism probes started")\n            refs[uid,role]=(image,z,scored)\n    write(output/"replay_gate.json",{"status":"PASSED","planned":16,"atol":ATOL,"rtol":RTOL})\n    for uid in IDS:\n        for role in ROLES:\n            image,z0,base_production=refs[uid,role]\n            for dx,dy in SHIFTS:\n                path=output/"records"/uid/role/(shift_id(dx,dy)+".json")\n                if path.exists() and read(path)["status"]=="SCORED": continue\n                attempt=output/"attempts"/uid/role/shift_id(dx,dy)/(uuid.uuid4().hex+".json")\n                write(attempt,{"status":"STARTED","time":time.time()})\n                try:\n                    zero=rgb_shift(image,dx,dy,False)\n                    circular=rgb_shift(image,dx,dy,True)\n                    same=bool(np.array_equal(np.asarray(zero),np.asarray(circular)))\n                    actual=z0 if (dx,dy)==(0,0) else reader.observe(zero)\n                    zcirc=actual if same else reader.observe(circular)\n                    production=base_production if (dx,dy)==(0,0) else reader.score(actual)\n                    fields,metrics=operator_fields(z0.float(),black,actual.float(),zcirc.float(),dx,dy)\n                    diagnostic={name:probe.score(z) for name,z in fields.items()}\n                    if (dx,dy)==(0,0):\n                        for name,z in fields.items():\n                            if relative(project(z),project(z0))>3e-5:\n                                raise RuntimeError(f"Zero-shift control differs: {name}")\n                    result={"status":"SCORED","rgb_shift_is_exact_integer":True,\n                            "rgb_periodic_zero_identical":same,"actual_production":production,\n                            "full_hf_actual":probe.score(actual,"full"),\n                            "full_hf_base":probe.score(z0,"full"),"hf_diagnostic":diagnostic,\n                            "metrics":metrics,"decomposition":decompose(diagnostic)}\n                    write(attempt,result)\n                    write(path,result)\n                    changes=result["decomposition"]["registered"]\n                    print(f"{uid[-4:]} {role} {shift_id(dx,dy)}: HF total={changes[\'total_loss\']:.5f}, phase={changes[\'phase_loss\']:.5f}, extra={changes[\'extra_vs_padded_phase\']:.5f}",flush=True)\n                except Exception as error:\n                    failed={"status":"OPERATIONAL_ERROR",**error_data(error)}\n                    write(attempt,failed);write(path,failed);summarize(output)\n                    raise\n            summarize(output)\n    final=summarize(output)\n    print(__import__("json").dumps({"status":final["status"],"counts":final["counts"]},indent=2),flush=True)\n\n\ndef main():\n    parser=argparse.ArgumentParser()\n    for name in ("source","output","repo"): parser.add_argument("--"+name,type=Path,required=True)\n    args=parser.parse_args()\n    if args.output.resolve().is_relative_to(args.source.resolve()) or args.source.resolve().is_relative_to(args.output.resolve()):\n        raise ValueError("SOURCE and OUTPUT must not overlap")\n    sys.path[:0]=[str(args.repo),str(args.repo/"src")]\n    try: run(args.source,args.output,args.repo)\n    except Exception as error:\n        write(args.output/"last_failure.json",error_data(error))\n        raise\n\n\nif __name__=="__main__": main()\n', encoding='utf-8')


In [ ]:
run_logged([sys.executable, str(script), '--source', str(SOURCE), '--output', str(output),
            '--repo', str(checkout)], 'mechanism-diagnostic', env=child_env, cwd=checkout)
summary = json.loads((output / 'summary.json').read_text())
print(json.dumps({'status':summary['status'], 'planned_records':summary['planned_records'],
                  'counts':summary['counts']}, indent=2))
print('Result directory:', output)


## 读数：先看对照，再判断机制

下图每组为相同 8 张图像：点为均值，竖线为样本最小—最大值，**不是置信区间**。
正负样本分开，不把同一图像的多次平移当作独立样本。完整 17 密钥分数在 `summary.json`。
数值恒等式：总损失 = 周期相位损失 + 边界模型差异 + 实际编码相对填充相位的额外差异。
这是有符号分数分解，**不是因果贡献比例**，负值保留。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
shifts = [(0,0),(-2,0),(2,0),(0,-2),(0,2),(8,0),(0,8)]
labels = ['0','x-2','x+2','y-2','y+2','x+8','y+8']
series = [('total_loss','Actual loss','#222222','o'),
          ('phase_loss','Periodic phase','#0072B2','s'),
          ('extra_vs_padded_phase','Extra vs padded phase','#D55E00','^')]
assert summary['status'] == 'COMPLETE_DIAGNOSTIC'
fig, axes = plt.subplots(2,2,figsize=(13,8),sharex=True,layout='constrained')
for i,role in enumerate(('positive','negative')):
    for j,metric in enumerate(('registered','gap')):
        ax=axes[i,j]
        for n,(field,label,color,marker) in enumerate(series):
            groups=[[r['decomposition'][metric][field] for r in summary['records']
                     if r['role']==role and (r['dx_rgb'],r['dy_rgb'])==shift] for shift in shifts]
            assert all(len(g)==8 for g in groups)
            values=np.asarray(groups)
            mean=values.mean(axis=1)
            ax.errorbar(np.arange(7)+(n-1)*.16,mean,
                        yerr=np.maximum(0,np.stack([mean-values.min(axis=1),values.max(axis=1)-mean])),
                        fmt=marker,color=color,label=label,capsize=2,linestyle='none')
        ax.axhline(0,color='gray',linewidth=.7)
        ax.set_title(f'{role}: {metric}, n=8 per shift')
        ax.set_ylabel('Signed common-HF score change')
        ax.set_xticks(range(7),labels)
        ax.grid(axis='y',alpha=.2)
        if i==1: ax.set_xlabel('Exact RGB shift (pixels)')
axes[0,0].legend(fontsize=9)
fig.suptitle('Saved Oracle images: conditional mechanism contrasts (not causal shares)')
fig.savefig(output / 'mechanism_contrasts.png',dpi=160)
plt.show()
# Fixed controls: show largest absolute score change per role; preserve every row in CSV.
controls=('boundary_model_delta','periodic_interpolation_delta','padding_32_vs_64',
          'rgb_boundary_delta','black_baseline_model_delta')
for role in ('positive','negative'):
    rows=[r for r in summary['records'] if r['role']==role]
    print(role, 'control max |registered-HF delta|:',
          {name:round(max(abs(r['decomposition']['registered'][name]) for r in rows),6) for name in controls})
    print(role, 'max removed residual energy fraction:',
          round(max(r['metrics']['nyquist_removed_energy_fraction_base_residual'] for r in rows),6))


## 可支持的结论与停止条件

- 如果纯相位已造成与实际路径接近的正确钥证据下降，而额外偏差较小、边界/插值/基线控制稳定，
  支持“当前 HF 读出相位敏感性是主要解释”。不能因此证明载体信息消失；纯相位本身可逆。
- 如果实际路径比相位及双线性模型都有明显额外下降，且已知逆移仍不能消除，支持存在
  “VAE 重编码链相对平移模型的额外非等变性”。这不等于证明 VAE 不可逆地删除水印。
- 如果 2 像素失败而 8 像素的额外偏差明显减小，提示子格点/下采样相位效应；若 8 像素仍有额外偏差，
  需考虑边界、非局部运算等，不能只归因于四分之一格点插值。
- 边界、插值、Nyquist 投影或黑基线假设若足以改变上述判断，结论记为 **未区分**；不继续增加搜索来制造阳性。
- 文件/哈希/版本/密钥不匹配、16 个旧分数重放失败、零平移或 Fourier 可逆性检查失败：立即停止并保留失败记录。
- 不自动训练、不改嵌入载体、不扩充图像、不重新估计 H、不把这一轮当作论文方法验证。

这里从保存 Oracle 图像再做整数平移，控制了新增 RGB 插值；因此不是逐像素复现上一轮“从裁剪图一次重采样”的实现。
既有 Oracle 重采样是所有路径共享的基线，本轮仅解释该基线附近的读出敏感性。

中断后保持 SOURCE/RUN_ID 不变运行全部单元；已完成记录跳过，重放检查会重做。
结果：`summary.json`、`readout.csv`、`replay_gate.json`、`protocol.json`、`environment.json`、
`records/`、`attempts/`、`execution_logs/`、`mechanism_contrasts.png`。不要同时写同一输出目录。
